In [ ]:
from IPython.display import display
import os
import re
import unicodedata
import warnings
import numpy as np
import pandas as pd
from google.colab import files
from sklearn.impute import SimpleImputer

warnings.filterwarnings("ignore")

# 1. Upload do arquivo CSV pelo usuário
uploaded = files.upload()
nome_arquivo = list(uploaded.keys())[0]

# 2. Detecção inteligente de cabeçalho deslocado e delimitador
def detectar_cabecalho_e_separador(caminho):
    separador = ";"
    header_idx = 0
    with open(caminho, "r", encoding="utf-8-sig", errors="ignore") as f:
        primeiras_linhas = [f.readline() for _ in range(15)]
    
    # Identifica separador predominante
    linhas_validas = [l for l in primeiras_linhas if l.strip()]
    if linhas_validas:
        primeira = linhas_validas[0]
        if primeira.count(";") >= primeira.count(","):
            separador = ";"
        else:
            separador = ","
            
    # Identifica a linha onde está o cabeçalho real
    for idx, l in enumerate(primeiras_linhas):
        if not l: break
        ll = l.lower()
        tem_estudante = any(k in ll for k in ["nome", "estudante", "aluno", "matricula", "matrícula"])
        tem_contexto = any(k in ll for k in ["turma", "escola", "q1", "p1", "idade", "sexo", "total", "responsavel", "observação", "observacao"])
        if tem_estudante and tem_contexto:
            header_idx = idx
            break
            
    return header_idx, separador

header_idx, sep_usado = detectar_cabecalho_e_separador(nome_arquivo)

# 3. Carrega na memória com proteção de formato
df = pd.read_csv(nome_arquivo, sep=sep_usado, skiprows=header_idx, encoding="utf-8-sig", dtype=str, low_memory=False)

print(f"✅ Arquivo '{nome_arquivo}' carregado com sucesso!")
if header_idx > 0:
    print(f"ℹ️ Cabeçalho detectado automaticamente na linha {header_idx + 1} (metadados iniciais ignorados).")
print(f"Linhas brutas: {df.shape[0]:,} | Colunas: {df.shape[1]}")


In [ ]:
# 1. Identifica itens de prova (Q1..Q50 ou P1..P40)
colunas_questoes = [c for c in df.columns if re.match(r"^[QP]\d+$", str(c).strip(), re.IGNORECASE)]
prefixo = colunas_questoes[0][0].upper() if colunas_questoes else "Q"
tipo_prova = "Vocabulário" if prefixo == "Q" else "TDE II"

# 2. Mapeamento robusto do nome do estudante (cobre todas as fases e variações)
coluna_nome = next((c for c in df.columns if any(p == c.strip().upper() for p in ["NOME", "NOME DO ESTUDANTE", "NOME ESTUDANTE", "NOME DO ALUNO", "ESTUDANTE", "ALUNO"])), None)
if coluna_nome is None:
    print("❌ ATENÇÃO: Nenhuma coluna de identificação de estudante foi encontrada.")
    print("Isso normalmente ocorre quando o arquivo é uma planilha de controle, chamada de turma ou diário de campo.")
    print(f"Colunas lidas: {list(df.columns[:8])}")
    raise ValueError(f"Arquivo sem coluna de estudantes. Verifique se este arquivo deve ser processado. Colunas disponíveis: {list(df.columns)}")

# 3. Mapeamento robusto da coluna de Escola (se não existir, infere pelo nome do arquivo)
coluna_escola = next((c for c in df.columns if "ESCOLA" in c.strip().upper()), None)
if coluna_escola is None:
    df["Escola"] = "Não Informado"
    if "prvl" in nome_arquivo.lower(): df["Escola"] = "EMEB PROFESSOR RICARDO VIEIRA DE LIMA"
    elif "pean" in nome_arquivo.lower(): df["Escola"] = "EMEB PADRE ANCHIETA"
    elif "pmqf" in nome_arquivo.lower(): df["Escola"] = "EMEB PROFESSORA MARIA QUEIROZ FERRO"
    elif "expedito" in nome_arquivo.lower(): df["Escola"] = "EMEB EXPEDITO PORFÍRIO DOS SANTOS"
    elif "mousinho" in nome_arquivo.lower(): df["Escola"] = "EMEF PADRE JOSÉ DOS SANTOS MOUSINHO"
    elif "natanael" in nome_arquivo.lower(): df["Escola"] = "EMEB NATANAEL DA SILVA"
    coluna_escola = "Escola"
elif coluna_escola != "Escola":
    df["Escola"] = df[coluna_escola]

# 4. Mapeamento robusto da coluna de Turma
coluna_turma = next((c for c in df.columns if "TURMA" in c.strip().upper()), None)
if coluna_turma and coluna_turma != "Turma":
    df["Turma"] = df[coluna_turma]

print(f"📋 Prova detectada: {tipo_prova} ({len(colunas_questoes)} itens, prefixo '{prefixo}')")
print(f"👤 Coluna de Estudante: '{coluna_nome}'")
print(f"🏫 Escola detectada: {df['Escola'].iloc[0] if 'Escola' in df.columns and len(df) > 0 else 'Não Informado'}")


In [ ]:
# 1. Elimina linhas fantasma (sem identificação do estudante)
linhas_antes = len(df)
df = df.dropna(subset=[coluna_nome]).copy()
df = df[df[coluna_nome].astype(str).str.strip() != ''].copy()

# 2. Deduplicação inteligente (mantém a linha com mais questões respondidas)
chave_pessoa = [c for c in ['Escola', coluna_nome, 'Idade'] if c in df.columns]
if chave_pessoa and colunas_questoes:
    df['_qtd_respostas'] = df[colunas_questoes].notnull().sum(axis=1)
    df = (
        df.sort_values(by='_qtd_respostas', ascending=False)
          .drop_duplicates(subset=chave_pessoa, keep='first')
          .sort_index()
          .drop(columns=['_qtd_respostas'])
          .copy()
    )

print(f"🧹 Linhas descartadas (fantasmas/duplicatas redundantes): {linhas_antes - len(df)}")
print(f"👥 Alunos únicos para tratamento: {len(df):,}")

In [ ]:
# Converte fórmulas quebradas herdadas do Excel (#REF!, #DIV/0!) para NaN real
ERROS_EXCEL = ['#REF!', '#DIV/0!', '#VALUE!', '#N/A', '#NAME?', '#NULL!', '#NUM!']
cols_check = colunas_questoes + (['Total'] if 'Total' in df.columns else [])
for col in cols_check:
    df[col] = df[col].replace(ERROS_EXCEL, np.nan)

# Identifica quem compareceu (tem pelo menos 1 resposta no teste)
fez_prova = df[colunas_questoes].notnull().any(axis=1)
df['fez_teste'] = fez_prova.astype(int)

# Imputação Psicométrica (CTT):
# - Alunos Faltosos (fez_teste == 0): NaN preservado (não leva zero injusto na média da coorte!)
# - Alunos Presentes (fez_teste == 1): questão em branco/rasurada vira 0.0; acerto vira 1.0 (ou 2.0 no TDE)
def limpar_questao(valor, presente, pref):
    if not presente:
        return np.nan
    if pd.isna(valor):
        return 0.0
    v = str(valor).strip().replace(',', '.')
    try:
        num = float(v)
        if pref == 'Q':
            return 1.0 if num == 1.0 else 0.0
        else:
            return num if num in (0.0, 1.0, 2.0) else 0.0
    except ValueError:
        return 0.0

for col in colunas_questoes:
    df[col] = [limpar_questao(v, p, prefixo) for v, p in zip(df[col], fez_prova)]

# Recalcular score Total real a partir das respostas (ignora Total corrompido do Excel)
df['Total_Original_Excel'] = pd.to_numeric(df.get('Total', pd.Series(dtype=float)), errors='coerce')
df['Total'] = np.nan
df.loc[fez_prova, 'Total'] = df.loc[fez_prova, colunas_questoes].sum(axis=1)

# Cria faixas de desempenho pedagógico
if prefixo == 'Q':
    bins, rotulos = [-1, 24, 34, 50], ['Baixo (0-24)', 'Médio (25-34)', 'Alto / Foi Bem (35-50)']
else:
    bins, rotulos = [-1, 39, 59, 80], ['Inferior (0-39)', 'Médio (40-59)', 'Superior (60-80)']

df['Nivel_Desempenho'] = pd.cut(df['Total'], bins=bins, labels=rotulos)
df['Nivel_Desempenho'] = df['Nivel_Desempenho'].cat.add_categories('Faltou / Não Avaliado').fillna('Faltou / Não Avaliado')

# Comparativo de Impacto Estatístico
media_bruta = df['Total_Original_Excel'].fillna(0).mean()
media_real = df.loc[fez_prova, 'Total'].mean()
print("=== IMPACTO DO TRATAMENTO DE DADOS (PREVENÇÃO DE VIÉS) ===")
print(f"Média com zero nos faltosos (Estratégia anterior): {media_bruta:.2f} acertos")
print(f"Média real apenas dos avaliados (Nova estratégia): {media_real:.2f} acertos")
print(f"Presentes: {fez_prova.sum():,} ({fez_prova.mean()*100:.1f}%) | Faltosos: {(~fez_prova).sum():,} ({(~fez_prova).mean()*100:.1f}%)")

df = df.drop(columns=['Total_Original_Excel'])
print("\nDistribuição por Nível de Desempenho:")
print(df['Nivel_Desempenho'].value_counts())

In [ ]:
def remover_acentos(texto):
    if pd.isna(texto):
        return ""
    texto = str(texto).strip().upper()
    return "".join(c for c in unicodedata.normalize("NFD", texto) if unicodedata.category(c) != "Mn")

# 1. PADRONIZAR ESCOLA
def padronizar_escola(val):
    t = remover_acentos(val)
    if not t or t in ["NAN", "NONE", "0", "0.0", "1.0"] or "/" in t:
        return "Não Informado"
    if "RICARDO VIEIRA" in t:
        return "EMEB PROFESSOR RICARDO VIEIRA DE LIMA"
    if "MARIA QUEIROZ" in t:
        return "EMEB PROFESSORA MARIA QUEIROZ FERRO"
    if "MOUSINHO" in t or "PADRE JOSE" in t:
        return "EMEF PADRE JOSÉ DOS SANTOS MOUSINHO"
    if "ANCHIETA" in t:
        return "EMEB PADRE ANCHIETA"
    if "EXPEDITO" in t:
        return "EMEB EXPEDITO PORFÍRIO DOS SANTOS"
    if "NATANAEL" in t:
        return "EMEB NATANAEL DA SILVA"
    return "Não Informado"

# 2. PADRONIZAR TURMA E DESAMBIGUAR "ANO REGULAR" vs "PERÍODO EJA"
def padronizar_turma(val):
    t = remover_acentos(val)
    if not t or t in ["NAN", "NONE", "0", "0.0"]:
        return "Não Informado"
    
    # Identifica expressamente Educação de Jovens e Adultos (EJA, Período, CAPIS, FIC)
    if any(k in t for k in ["PERIODO", "EJA", "CAPIS", "FIC"]):
        m_per = re.search(r"([1-9])\s*[º°ª]?\s*PERIODO", t)
        if m_per:
            return f"{m_per.group(1)}º PERÍODO (EJA)"
        return "EJA"
    
    # Ensino Fundamental Regular (5º ao 9º Ano)
    t_limpo = re.sub(r"\b(MATUTINO|VESPERTINO|MAT|VESP|2023|2024)\b", " ", t)
    t_limpo = re.sub(r"\s+", " ", t_limpo).strip()
    m_ano_letra = re.search(r"([5-9])\s*[º°ª]?\s*(?:ANO)?\s*[-–]?\s*([A-F])\b", t_limpo)
    if m_ano_letra:
        return f"{m_ano_letra.group(1)}º ANO {m_ano_letra.group(2)}"
    m_so_ano = re.search(r"([5-9])\s*[º°ª]?\s*ANO\b", t_limpo)
    if m_so_ano:
        return f"{m_so_ano.group(1)}º ANO A"
    return t_limpo

# 3. PADRONIZAR SEXO
def padronizar_sexo(val):
    t = remover_acentos(val)
    return "M" if t.startswith("M") else ("F" if t.startswith("F") else "Não Informado")

# 4. PADRONIZAR ZONA
def padronizar_zona(val):
    t = remover_acentos(val)
    if not t or t in ["NAN", "NONE", "NAO INFORMADO", "0", "0.0", "-"]:
        return "Não Informado"
    if "INDIG" in t:
        return "Indígena"
    if "QUILOMB" in t:
        return "Quilombola"
    if any(p in t for p in ["RURAL", "POV", "PV", "SITIO", "FAZENDA", "TERRA NOVA", " GADO", "CANA BRAVA"]):
        return "Rural"
    if any(p in t for p in ["URBANA", "CENTRO", "BAIRRO", "RUA", "CRUZEIRO", "RANCHO", "SAO JOSE"]):
        return "Urbana"
    return "Não Informado"

# Aplicação das padronizações
if "Escola" in df.columns: df["Escola"] = df["Escola"].apply(padronizar_escola)
if "Turma" in df.columns:
    df["Turma"] = df["Turma"].apply(padronizar_turma)
    # Engenharia de Atributo: Modalidade de Ensino explícita para evitar confusão na IA
    df["Modalidade_Ensino"] = df["Turma"].apply(lambda x: "EJA" if "EJA" in str(x).upper() or "PERÍODO" in str(x).upper() else "Regular")
    df["Serie_Ano"] = df["Turma"].apply(lambda x: "EJA (Períodos)" if "PERÍODO" in str(x).upper() or "EJA" in str(x).upper() else str(x)[:6].strip())
if "Sexo" in df.columns: df["Sexo"] = df["Sexo"].apply(padronizar_sexo)
if "Zona" in df.columns:
    df["Zona"] = df["Zona"].apply(padronizar_zona)
    df["zona_informada"] = (df["Zona"] != "Não Informado").astype(int)

print("✅ Escolas, Turmas, Sexo, Zonas e Modalidade de Ensino padronizados com sucesso!")
print("Distribuição por Modalidade de Ensino:")
print(df["Modalidade_Ensino"].value_counts())


In [ ]:
# 1. Conversão e derivação de Idade a partir de Data Nascimento
if "Data Nascimento" in df.columns:
    dt = pd.to_datetime(df["Data Nascimento"], dayfirst=True, format="mixed", errors="coerce")
    df["Data Nascimento"] = dt.dt.strftime("%d/%m/%Y")
    idade_calc = 2024 - dt.dt.year
    df["Idade"] = idade_calc.fillna(pd.to_numeric(df.get("Idade", np.nan), errors="coerce"))
else:
    df["Idade"] = pd.to_numeric(df.get("Idade", np.nan), errors="coerce")

# Limpa idades com erro biológico (< 6 ou > 80)
df.loc[(df["Idade"] < 6) | (df["Idade"] > 80), "Idade"] = np.nan
df["idade_informada"] = df["Idade"].notna().astype(int)

# 2. Imputação Dinâmica (Calculada exclusivamente a partir dos dados do próprio arquivo aberto)
if df["Idade"].notna().any():
    # Nível 1: Mediana real calculada para cada turma específica deste arquivo
    df["Idade"] = df.groupby("Turma")["Idade"].transform(lambda x: x.fillna(x.median()))
    # Nível 2: Mediana real calculada para cada modalidade (EJA vs Regular) deste arquivo
    df["Idade"] = df.groupby("Modalidade_Ensino")["Idade"].transform(lambda x: x.fillna(x.median()))
    # Nível 3: Mediana geral observada nos dados deste arquivo
    df["Idade"] = df["Idade"].fillna(df["Idade"].median()).astype(int)
    print("✅ Imputação dinâmica de idade concluída com base nos dados deste arquivo!")
else:
    # Caso extremo: o arquivo inteiro veio 100% sem idade
    df["Idade"] = np.where(df["Modalidade_Ensino"] == "EJA", 25, 13)
    print("Tabela 100% sem coluna de idade: aplicada referência padrão por modalidade.")

# 3. Cálculo Dinâmico da Distorção Idade-Série (Conforme padrão oficial MEC: 2+ anos de atraso)
# Para cada ano escolar (5º ao 9º), calcula a idade esperada daquela série específica:
# 5º ano -> esperada: 10 anos | 6º ano -> 11 anos | 7º ano -> 12 anos | 8º ano -> 13 anos | 9º ano -> 14 anos
def checar_distorcao(row):
    if row["Modalidade_Ensino"] != "Regular" or pd.isna(row["Idade"]):
        return 0
    m = re.search(r"([5-9])\\s*[º°ª]?\\s*ANO", str(row.get("Turma", "")).upper())
    if m:
        ano_escolar = int(m.group(1))
        idade_esperada = ano_escolar + 5
        # Critério MEC: atraso escolar significativo se idade >= idade esperada + 2
        return 1 if row["Idade"] >= (idade_esperada + 2) else 0
    return 1 if row["Idade"] > 16 else 0

df["distorcao_idade_serie"] = df.apply(checar_distorcao, axis=1)

# 4. Preservação de Atributos Sensíveis para LGPD e Fairness
if "Cor/Raça" in df.columns:
    df["Cor/Raça"] = df["Cor/Raça"].astype(str).str.strip().replace({"0": "Não Informado", "nan": "Não Informado"})
if "Necessidade/Deficiência" in df.columns:
    df["Necessidade/Deficiência"] = df["Necessidade/Deficiência"].fillna("Sem Necessidade Informada")

# Diagnóstico impresso na tela específico deste arquivo
print(f"Estatísticas observadas neste arquivo:")
if (df["Modalidade_Ensino"] == "Regular").any():
    print(f"  - Mediana de Idade no Regular: {df[df['Modalidade_Ensino']=='Regular']['Idade'].median():.0f} anos")
if (df["Modalidade_Ensino"] == "EJA").any():
    print(f"  - Mediana de Idade na EJA:     {df[df['Modalidade_Ensino']=='EJA']['Idade'].median():.0f} anos")
print(f"  - Alunos do Regular com distorção idade-série: {df['distorcao_idade_serie'].sum():,} de {len(df):,}")


In [ ]:
# Desativa o corte de linhas e de colunas no Pandas
pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)

# Exibe a tabela completa com todas as linhas e colunas tratadas
display(df)


In [ ]:
from google.colab import files

# 1. Define o nome do arquivo tratado (se 'nome_arquivo' existir, adiciona 'tratado_' na frente)
nome_saida = f"tratado_{os.path.basename(nome_arquivo)}" if 'nome_arquivo' in locals() else "tabela_tratada.csv"

# 2. Salva em CSV mantendo os acentos em português (utf-8-sig) e separador ';'
df.to_csv(nome_saida, sep=";", index=False, encoding="utf-8-sig")

# 3. Dispara o download automático no seu navegador
files.download(nome_saida)

print(f" Arquivo '{nome_saida}' salvo e baixado com sucesso!")
